# 📰 Fake News Detection Using Deep Learning

Builds, trains and compares three deep-learning models — **LSTM**, **GRU** and a **fine-tuned BERT** — that classify news articles as **REAL** or **FAKE**.

Everything runs on **PyTorch** (one framework, one GPU user → no TensorFlow/PyTorch conflicts that crash Colab).

**Pipeline:** setup → load data → EDA → cleaning → split → vocabulary + GloVe embeddings → LSTM → GRU → BERT → evaluation & comparison → predict on your own text → save models.

> 💡 Use **Runtime → Change runtime type → GPU (T4)**, then **Runtime → Run all** once (no restarts in between).


## 1. Setup

In [ ]:
# Colab already ships with compatible torch/numpy/pandas. Only install what is missing — do NOT use --upgrade.
!pip install -q transformers nltk
print("Installs complete.")


In [ ]:
import os, re, gc, json, pickle, random, warnings, zipfile, urllib.request
from collections import Counter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from transformers import BertTokenizerFast, BertForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 2. Load the Dataset
The public **"Fake or Real News"** dataset (~6,300 labelled articles) is downloaded straight from GitHub — no upload or API key needed.
To use your own data, replace `DATA_URL` with a CSV path and adjust `TEXT_COL` / `LABEL_COL`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv"
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
df.head()


In [ ]:
TEXT_COL, LABEL_COL = "text", "label"

df = df.dropna(subset=[TEXT_COL, LABEL_COL]).drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
df["full_text"] = (df["title"].fillna("") + ". " + df[TEXT_COL].fillna("")) if "title" in df.columns else df[TEXT_COL]

df[LABEL_COL] = df[LABEL_COL].astype(str).str.upper().str.strip()
df["target"] = df[LABEL_COL].map({"FAKE": 1, "REAL": 0})      # 1 = fake (positive class)
df = df.dropna(subset=["target"]).reset_index(drop=True)
df["target"] = df["target"].astype(int)
print("Final shape:", df.shape)
df[["full_text", LABEL_COL, "target"]].head()


## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=LABEL_COL, data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Class Distribution"); axes[0].set_xlabel("")
df["text_len"] = df["full_text"].apply(lambda x: len(str(x).split()))
sns.histplot(data=df, x="text_len", hue=LABEL_COL, bins=50, kde=True, ax=axes[1])
axes[1].set_xlim(0, 1500); axes[1].set_title("Article Length (words)")
plt.tight_layout(); plt.show()
print(df["text_len"].describe())


## 4. Text Cleaning
Lowercase → remove URLs/emails → strip punctuation & digits → tokenize → drop stopwords and very short tokens. (Used for LSTM/GRU; BERT uses the raw text.)

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in word_tokenize(text) if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

df["clean_text"] = df["full_text"].apply(clean_text)
df[["full_text", "clean_text"]].sample(3, random_state=SEED)


## 5. Train / Validation / Test Split
We split **row indices once**, so LSTM, GRU and BERT are all evaluated on exactly the same test articles.

In [ ]:
idx = np.arange(len(df))
y_all = df["target"].values
idx_train, idx_tmp = train_test_split(idx, test_size=0.3, random_state=SEED, stratify=y_all)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED, stratify=y_all[idx_tmp])

clean_all, raw_all = df["clean_text"].values, df["full_text"].values
X_train, X_val, X_test = clean_all[idx_train], clean_all[idx_val], clean_all[idx_test]            # cleaned (LSTM/GRU)
R_train, R_val, R_test = raw_all[idx_train], raw_all[idx_val], raw_all[idx_test]                  # raw (BERT)
y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]
print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")


## 6. Vocabulary & Padding (LSTM / GRU)
We build the vocabulary from the training set only and pad/truncate every article to a fixed length.

In [ ]:
MAX_VOCAB, MAX_LEN = 20000, 300

counter = Counter(w for t in X_train for w in t.split())
word_index = {"<PAD>": 0, "<OOV>": 1}
for i, (w, _) in enumerate(counter.most_common(MAX_VOCAB - 2), start=2):
    word_index[w] = i
vocab_size = len(word_index)

def encode(texts):
    seqs = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [word_index.get(w, 1) for w in t.split()][:MAX_LEN] or [1]
        seqs[i, :len(ids)] = ids
        lengths[i] = len(ids)
    return seqs, lengths

Xtr_seq, Ltr = encode(X_train)
Xva_seq, Lva = encode(X_val)
Xte_seq, Lte = encode(X_test)
print("Vocab size:", vocab_size, "| train matrix:", Xtr_seq.shape)


## 7. GloVe Word Embeddings
Pretrained 100-d GloVe vectors initialise the embedding layer (words not in GloVe get small random vectors).

In [ ]:
GLOVE_DIR = "/content/glove"; os.makedirs(GLOVE_DIR, exist_ok=True)
GLOVE_ZIP = os.path.join(GLOVE_DIR, "glove.6B.zip")
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.100d.txt")
EMBED_DIM = 100

if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe (about 1-2 minutes)...")
    urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", GLOVE_ZIP)
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)
    os.remove(GLOVE_ZIP)
    print("Done.")


In [ ]:
rng = np.random.RandomState(SEED)
embedding_matrix = rng.normal(0, 0.1, (vocab_size, EMBED_DIM)).astype("float32")
embedding_matrix[0] = 0.0   # padding row

hits = 0
with open(GLOVE_FILE, encoding="utf8") as f:          # stream the file: no giant dictionary kept in RAM
    for line in f:
        word, *vec = line.rstrip().split(" ")
        i = word_index.get(word)
        if i is not None and i > 1:
            embedding_matrix[i] = np.asarray(vec, dtype="float32")
            hits += 1
print(f"GloVe coverage: {hits}/{vocab_size} words ({hits/vocab_size:.1%})")


## 8. LSTM & GRU Models
A 2-layer **bidirectional** recurrent network over the GloVe embeddings; the final hidden states feed a small classifier head.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, embedding_matrix=None, rnn_type="lstm", hidden=64, dropout=0.4):
        super().__init__()
        self.rnn_type = rnn_type
        if embedding_matrix is not None:
            self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        rnn_cls = nn.LSTM if rnn_type == "lstm" else nn.GRU
        self.rnn = rnn_cls(embed_dim, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if self.rnn_type == "lstm":
            h = h[0]                                   # (h_n, c_n) -> h_n
        last = torch.cat([h[-2], h[-1]], dim=1)        # last layer: forward + backward states
        return self.head(last).squeeze(1)              # raw logit (P(fake) = sigmoid)

def make_loader(X, L, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.tensor(X), torch.tensor(L), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader_rnn = make_loader(Xtr_seq, Ltr, y_train, shuffle=True)
val_loader_rnn = make_loader(Xva_seq, Lva, y_val, shuffle=False, batch_size=128)
test_loader_rnn = make_loader(Xte_seq, Lte, y_test, shuffle=False, batch_size=128)

def run_rnn_epoch(model, loader, loss_fn, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for xb, lb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(xb, lb)
            loss = loss_fn(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        tot_loss += loss.item() * yb.size(0)
        correct += ((logits > 0).float() == yb).sum().item()
        n += yb.size(0)
    return tot_loss / n, correct / n

def train_rnn(rnn_type, epochs=15, patience=3, lr=1e-3):
    model = RNNClassifier(vocab_size, EMBED_DIM, embedding_matrix, rnn_type).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    hist = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        tl, ta = run_rnn_epoch(model, train_loader_rnn, loss_fn, optimizer)
        vl, va = run_rnn_epoch(model, val_loader_rnn, loss_fn)
        for k, v in zip(hist, (tl, vl, ta, va)):
            hist[k].append(v)
        print(f"[{rnn_type.upper()}] epoch {ep:02d} | loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f}")
        if vl < best_val:
            best_val, wait = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping."); break
    model.load_state_dict(best_state)      # restore best weights
    return model, hist


### LSTM

In [ ]:
lstm_model, lstm_hist = train_rnn("lstm")


### GRU

In [ ]:
gru_model, gru_hist = train_rnn("gru")


In [ ]:
def plot_history(h, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss"); axes[0].legend()
    axes[1].plot(h["accuracy"], label="train"); axes[1].plot(h["val_accuracy"], label="val")
    axes[1].set_title(f"{title} — Accuracy"); axes[1].legend()
    plt.tight_layout(); plt.show()

plot_history(lstm_hist, "LSTM")
plot_history(gru_hist, "GRU")


## 9. Fine-tuned BERT
`bert-base-uncased` fine-tuned with 🤗 Transformers + PyTorch. BERT has its own WordPiece tokenizer and works best on **raw** text, so it uses the uncleaned articles.

In [ ]:
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LEN, BERT_BATCH_SIZE, BERT_EPOCHS = 128, 16, 2

bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = bert_tokenizer(str(self.texts[i]), truncation=True, padding="max_length",
                             max_length=BERT_MAX_LEN, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[i], dtype=torch.long)}

train_loader = DataLoader(NewsDataset(R_train, y_train), batch_size=BERT_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NewsDataset(R_val, y_val), batch_size=BERT_BATCH_SIZE)
test_loader = DataLoader(NewsDataset(R_test, y_test), batch_size=BERT_BATCH_SIZE)
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bert_model = BertForSequenceClassification.from_pretrained(BERT_MODEL_NAME, num_labels=2).to(DEVICE)
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * BERT_EPOCHS)

def run_bert_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    for batch in loader:
        ids, mask, labels = (batch[k].to(DEVICE) for k in ("input_ids", "attention_mask", "labels"))
        with torch.set_grad_enabled(train):
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            if train:
                optimizer.zero_grad()
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
        tot_loss += out.loss.item() * labels.size(0)
        correct += (out.logits.argmax(1) == labels).sum().item()
        n += labels.size(0)
    return tot_loss / n, correct / n

for ep in range(1, BERT_EPOCHS + 1):
    tl, ta = run_bert_epoch(bert_model, train_loader, train=True)
    vl, va = run_bert_epoch(bert_model, val_loader, train=False)
    print(f"[BERT] epoch {ep}/{BERT_EPOCHS} | loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f}")


## 10. Evaluation on the Test Set
Accuracy, precision, recall and F1 for all three models on the **same** held-out articles (positive class = FAKE).

In [ ]:
@torch.no_grad()
def probs_rnn(model, loader):
    model.eval()
    out = []
    for xb, lb, _ in loader:
        out.append(torch.sigmoid(model(xb.to(DEVICE), lb)).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def probs_bert(model, loader):
    model.eval()
    out = []
    for batch in loader:
        logits = model(input_ids=batch["input_ids"].to(DEVICE), attention_mask=batch["attention_mask"].to(DEVICE)).logits
        out.append(torch.softmax(logits, 1)[:, 1].cpu().numpy())
    return np.concatenate(out)

def summarize(name, probs):
    preds = (probs >= 0.5).astype(int)
    return {"model": name, "accuracy": accuracy_score(y_test, preds), "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds), "f1": f1_score(y_test, preds),
            "y_true": y_test, "y_pred": preds, "y_prob": probs}

results = [summarize("LSTM", probs_rnn(lstm_model, test_loader_rnn)),
           summarize("GRU", probs_rnn(gru_model, test_loader_rnn)),
           summarize("BERT", probs_bert(bert_model, test_loader))]

comparison_df = pd.DataFrame(results)[["model", "accuracy", "precision", "recall", "f1"]]
comparison_df


In [ ]:
ax = comparison_df.set_index("model").plot(kind="bar", figsize=(8, 5), rot=0, colormap="viridis")
ax.set_title("Model Comparison"); ax.set_ylim(0, 1); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, r in zip(axes, results):
    sns.heatmap(confusion_matrix(r["y_true"], r["y_pred"]), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"])
    ax.set_title(f"{r['model']} — Confusion Matrix"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()


In [ ]:
plt.figure(figsize=(7, 6))
for r in results:
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_prob"])
    plt.plot(fpr, tpr, label=f"{r['model']} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curves")
plt.legend(loc="lower right"); plt.show()


In [ ]:
for r in results:
    print(f"\n===== {r['model']} — Classification Report =====")
    print(classification_report(r["y_true"], r["y_pred"], target_names=["REAL", "FAKE"]))


## 11. Try It Yourself
Run any headline or article through the best-scoring model.

In [ ]:
best_model_name = comparison_df.sort_values("f1", ascending=False).iloc[0]["model"]
print("Best model by F1:", best_model_name)

@torch.no_grad()
def predict_news(text, model_name=best_model_name):
    if model_name == "BERT":
        bert_model.eval()
        enc = bert_tokenizer(text, truncation=True, padding="max_length", max_length=BERT_MAX_LEN,
                             return_tensors="pt").to(DEVICE)
        p_fake = torch.softmax(bert_model(**enc).logits, 1)[0, 1].item()
    else:
        model = lstm_model if model_name == "LSTM" else gru_model
        seq, length = encode([clean_text(text)])
        model.eval()
        p_fake = torch.sigmoid(model(torch.tensor(seq).to(DEVICE), torch.tensor(length))).item()
    label = "FAKE" if p_fake >= 0.5 else "REAL"
    conf = p_fake if label == "FAKE" else 1 - p_fake
    print(f"Model: {model_name} | Prediction: {label} (confidence {conf:.1%})")
    return label, conf

predict_news("Scientists at a major university confirm breakthrough in renewable energy storage technology.")


## 12. Save Models
Everything the Streamlit app needs is written to `/content/saved_models/`.

In [ ]:
SAVE_DIR = "/content/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

for name, model in (("lstm", lstm_model), ("gru", gru_model)):
    torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "vocab_size": vocab_size, "embed_dim": EMBED_DIM, "rnn_type": name, "hidden": 64},
               os.path.join(SAVE_DIR, f"{name}_model.pt"))

with open(os.path.join(SAVE_DIR, "vocab.pkl"), "wb") as f:
    pickle.dump(word_index, f)
with open(os.path.join(SAVE_DIR, "config.json"), "w") as f:
    json.dump({"max_len": MAX_LEN, "bert_max_len": BERT_MAX_LEN}, f)

bert_model.save_pretrained(os.path.join(SAVE_DIR, "bert_model"))
bert_tokenizer.save_pretrained(os.path.join(SAVE_DIR, "bert_model"))
print("✅ Saved to", SAVE_DIR, "->", os.listdir(SAVE_DIR))


In [ ]:
# Optional: zip the folder so you can download it from the Colab file browser (left sidebar → Files)
!cd /content && zip -rq saved_models.zip saved_models && ls -lh saved_models.zip


## 13. Conclusion & Next Steps
- LSTM, GRU and fine-tuned BERT were trained on the same split and compared on accuracy, precision, recall, F1 and ROC-AUC.
- BERT usually wins thanks to contextual embeddings and large-scale pre-training, at the cost of more compute; LSTM/GRU are lighter and faster to deploy.
- **Caveat:** the dataset is mostly 2016-era US political news, so the models learn *writing-style patterns*, not facts, and may not generalise to other topics or periods.

**Ideas:** LIME/SHAP explanations, more BERT epochs, larger/newer datasets (LIAR, ISOT), a Streamlit/FastAPI deployment.